In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf master.zip

# 2. 克隆仓库
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    
    # 4. 进入目录
    %cd Diffusion-Illusions
    
    # 5. 安装依赖
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
    
else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os
repo_name = "Diffusion-Illusions"
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
import torch
import torch.nn as nn
import rp
from typing import Tuple
import source.stable_diffusion as sd
from source.stable_diffusion_labels import SimpleLabel
from source.learnable_textures import LearnableImageFourier, LearnableImageRasterSigmoided

# --- 辅助函数 ---
def rgb_to_grayscale(rgb: torch.Tensor) -> torch.Tensor:
    weights = torch.tensor([0.299, 0.587, 0.114], device=rgb.device, dtype=rgb.dtype)
    if len(rgb.shape) == 3:
        gray = (rgb * weights[:, None, None]).sum(dim=0, keepdim=True)
    else:
        gray = (rgb * weights[None, :, None, None]).sum(dim=1, keepdim=True)
    return gray

def grayscale_to_rgb(gray: torch.Tensor) -> torch.Tensor:
    if len(gray.shape) == 3:
        return gray.repeat(3, 1, 1)
    else:
        return gray.repeat(1, 3, 1, 1)

# --- 核心模型类 ---
class LearnableColorHybrid(nn.Module):
    def __init__(self, size: int = 512, representation: str = 'fourier'):
        super().__init__()
        # 只使用一个可学习的 RGB 图像
        if representation == 'fourier':
            self.learnable_image = LearnableImageFourier(size, size, 3)
        else:
            self.learnable_image = LearnableImageRasterSigmoided(size, size, 3)
            
    def forward(self) -> Tuple[torch.Tensor, torch.Tensor]:
        img = self.learnable_image()
        # 实时计算灰度视图
        gray = rgb_to_grayscale(img)
        gray_view = grayscale_to_rgb(gray)
        return img, gray_view

In [ ]:
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    print("✅ 模型加载完毕！")
else:
    print("模型已存在。")

In [ ]:
from IPython.display import clear_output

# ===========================
# 🔧 参数配置
# ===========================
# 1. 彩色模式下看到的内容
prompt_color = "a peaceful mountain landscape with lake, oil painting style"
# 2. 灰度/黑白模式下看到的内容
prompt_grayscale = "a fierce tiger with black stripes, close up portrait"

NUM_ITER = 3000
LEARNING_RATE = 1e-4
GUIDANCE_SCALE = 100
DISPLAY_INTERVAL = 100

# ===========================
# 🚀 训练循环
# ===========================
print(f"初始化任务: 彩色[{prompt_color}] vs 灰度[{prompt_grayscale}]")

model = LearnableColorHybrid(size=512, representation='fourier').to(device)
label_c = SimpleLabel(prompt_color)
label_g = SimpleLabel(prompt_grayscale)
optim = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

display_eta = rp.eta(NUM_ITER, title='Training Status')

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)
        
        # 获取当前图像及其灰度版本
        rgb_img, gray_view = model()
        
        # Loss 1: 彩色图应该像 prompt_color
        model_sd.train_step(label_c.embedding, rgb_img[None], guidance_scale=GUIDANCE_SCALE)
        
        # Loss 2: 灰度图应该像 prompt_grayscale
        model_sd.train_step(label_g.embedding, gray_view[None], guidance_scale=GUIDANCE_SCALE)
        
        optim.step()
        optim.zero_grad()
        
        if iter_num % DISPLAY_INTERVAL == 0:
            clear_output(wait=True)
            with torch.no_grad():
                curr_rgb, curr_gray = model()
                vis_rgb = rp.as_numpy_image(curr_rgb)
                vis_gray = rp.as_numpy_image(curr_gray)
                
                print(f"Iteration {iter_num} / {NUM_ITER}")
                print(f"左: 彩色模式 | 右: 灰度模式")
                # 并排显示
                rp.display_image(rp.tiled_images([vis_rgb, vis_gray]))

except KeyboardInterrupt:
    print("⚠️ 用户停止训练")

# 显示最终结果
print("✅ 最终结果 (右键保存):")
with torch.no_grad():
    final_rgb, _ = model()
    rp.display_image(rp.as_numpy_image(final_rgb))